# Sets

In [75]:
import sys
from pathlib import Path

current = Path.cwd()
for parent in [current, *current.parents]:
    if (parent / '_config.yml').exists():
        project_root = parent  # ← Add project root, not chapters
        break
else:
    project_root = Path.cwd().parent.parent

sys.path.insert(0, str(project_root))

from shared import thinkpython, diagram, jupyturtle
from shared.download import download

# Register as top-level modules so direct imports work in subsequent cells
sys.modules['thinkpython'] = thinkpython
sys.modules['diagram'] = diagram
sys.modules['jupyturtle'] = jupyturtle


## What is a set

A set is an **unordered** collection of unique values. Unlike lists or tuples, sets do not allow **duplicates**, and the elements have no guaranteed order. 

Here below is an example of a set. Note that it uses curly braces just like dictionaries. 

In [76]:
s = {1, 2, 3, 3, 2}
print(s)                ### {1, 2, 3}

{1, 2, 3}


Sets are useful when you care about **membership**: whether something is **in** a collection; rather than order or position.

The most common use cases are:

- Removing duplicates from a list
- Checking membership quickly
- Comparing two collections (what's shared, what's different)

Comparing Sets, Lists, and Dicts


| Feature      | Set                    | List          | Dict                          |
|--------------|------------------------|---------------|-------------------------------|
| Ordered      | No                     | Yes           | Yes (insertion order)         |
| Duplicates   | No                     | Yes           | Keys: No, Values: Yes         |
| Mutable      | Yes                    | Yes           | Yes                           |
| Allows only hashable elements | Yes   | No            | Keys only                     |
| Lookup speed | Avg O(1)               | O(n)          | Avg O(1)                      |
| Use case     | Unique items, membership | Ordered items | Key-value pairs               |

The four key properties that make sets different from other data structures:

1. **Unordered**: Elements have no guaranteed position. There is no first or last element.
2. **Unique**: Duplicates are automatically removed.
3. **Mutable**: You can add and remove elements.
4. **Hashable elements only**: Every element must be hashable. In practice this means immutable types: `int`, `float`, `str`, `tuple`.

In [77]:
### 1. Sets are unordered, so the order of elements is not guaranteed.
{3, 1, 2} == {1, 2, 3}  # True

True

In [78]:
### 2. Sets do not allow duplicate elements, so duplicates are automatically removed.
{1, 2, 2, 3, 3}         # {1, 2, 3}

{1, 2, 3}

In [79]:
%%expect TypeError

### 3. Sets are mutable, so you can add or remove elements after the set is created.
s = {1, 2, 3}
s.add(4)                # works
s.add([5, 6])           # TypeError

TypeError: unhashable type: 'list'

In [80]:
%%expect TypeError

### 4. Sets can contain only hashable (immutable) elements, so you cannot include lists or other sets as elements.
{1, "hello", (1, 2)}    # valid
{[1, 2], [3, 4]}        # TypeError

TypeError: unhashable type: 'list'

### Hashable vs mutable

- **Hashable** means an object has a hash value that stays stable during its lifetime.
- **Mutable** objects usually are **not hashable** (for example: `list`, `dict`, `set`).
- **Immutable** objects are often hashable (for example: `int`, `float`, `str`, `tuple`), but for tuples, all nested elements must also be hashable.
- For hash-table types: `dict` requires **hashable keys**, and `set` requires **hashable elements**.

| Method                              | List | Dict | Set | Tuple | String |
|-------------------------------------|------|------|-----|-------|--------|
| `.append(x)`                        | ✓    |      |     |       |        |
| `.insert(i, x)`                     | ✓    |      |     |       |        |
| `.extend(iter)`                     | ✓    |      |     |       |        |
| `.update(other)`                    |      | ✓    | ✓   |       |        |
| `.remove(x)`                        | ✓    |      | ✓   |       |        |
| `.pop()`                            | ✓    | ✓    | ✓   |       |        |
| `.popitem()`                        |      | ✓    |     |       |        |
| `.add(x)`                           |      |      | ✓   |       |        |
| `.discard(x)`                       |      |      | ✓   |       |        |
| `.sort()`                           | ✓    |      |     |       |        |
| `.reverse()`                        | ✓    |      |     |       |        |
| `.setdefault(k, v)`                 |      | ✓    |     |       |        |
| `.clear()`                          | ✓    | ✓    | ✓   |       |        |
| `.intersection_update(t)`           |      |      | ✓   |       |        |
| `.difference_update(t)`             |      |      | ✓   |       |        |
| `.symmetric_difference_update(t)`   |      |      | ✓   |       |        |

## Creating sets

A set is created using curly braces `{}` with comma-separated values, or by using the `set()` constructor. 

Unlike lists, sets are unordered and do not allow duplicates - if you add the same value twice, it only appears once. This makes sets useful for storing **unique** items. One important gotcha: an empty {} creates a dict, not a set - you must use set() for an empty set.


In [81]:
s = {1, 2, 3, 3}       # duplicate 3 is ignored
print(s)               # {1, 2, 3}

s = set([1, 2, 2, 3])  # from a list
print(s)               # {1, 2, 3}

s = set()              # empty set
d = {}                 # this is a dict, not a set!

{1, 2, 3}
{1, 2, 3}


Set elements must be hashable. Since sets use **hashing** internally to enforce uniqueness and enable fast lookup, all elements must be hashable. This means you can store integers, floats, strings, and tuples in a set, but not lists or dicts.

In [82]:
%%expect TypeError

# Valid
s = {1, "hello", (1, 2)}

# Invalid
s = {[1, 2], [3, 4]}    # TypeError: unhashable type: 'list'

TypeError: unhashable type: 'list'

A tuple is hashable only if **all** of its elements are hashable.

In [83]:
print(hash((1, 2)))          # ok
print(hash((1, (2, 3))))     # ok

try:
    hash((1, [2, 3]))        # list inside tuple => unhashable
except TypeError as e:
    print(e)

-3550055125485641917
7267574591690527098
unhashable type: 'list'


### Accessing Elements

Sets have no indexing or positional access. Because sets are unordered, there's no concept of "first" or "second" element, so index access makes no sense. 

Compared with list and dictionary, `dict` is the only one with a built-in safe getter. A `list` requires a **guard clause**. 

| | List | Dict | Set |
|---|---|---|---|
| **Access** | by index `l[i]` | by key `d[key]` | no direct access |
| **Safe get** | `l[i] if i < len(l) else default` | `d.get(key, default)` | `val if val in s else default` |
| **Built-in safe method** | none | `.get()` | none |

In [84]:
%%expect IndexError

d = {'x': 1, 'y': 2}
d.get('z', 'not found')                 # 'not found' — no error

l = [10, 20, 30]
l[5]                                  # IndexError!   ### uncomment to see the error

IndexError: list index out of range

A safe way to handle a possible out-of-range error is to use a guard clause. 

In [85]:
l = [10, 20, 30]

result = l[5] if 5 < len(l) else None   # None — safe
print(result)                           # None

None


To work with elements in a set, you can:

| Method | Example | Notes |
|---|---|---|
| **Membership check** | `3 in s` | True/False only, no error |
| **Iterate** | `for x in s:` | no guaranteed order |
| **Convert to list** | `list(s)[0]` | order not guaranteed |
| **`s.pop()`** | `s.pop()` | returns an arbitrary element |
| **Direct index** | `s[0]` | TypeError, not supported |

In [86]:
### check membership

if "apple" in s:
    print("found")

## Set Operations 

Sets support mathematical operations that make it easy to compare two collections.

Each operation has both an operator shorthand and a method form — they produce the same result. A quick reference:

| Operation            | Operator | Method                       |
|----------------------|----------|------------------------------|
| Union                | `a \| b`  | `a.union(b)`                 |
| Intersection         | `a & b`  | `a.intersection(b)`          |
| Difference           | `a - b`  | `a.difference(b)`            |
| Symmetric Difference | `a ^ b`  | `a.symmetric_difference(b)`  |

In [87]:
a = {1, 2, 3}
b = {3, 4, 5}

Operator forms (`|`, `&`, `-`, `^`) require set operands. Method forms are more flexible and can take any iterable.

In [88]:
print(a.union([3, 4, 5]))    # works: method accepts iterable

try:
    a | [3, 4, 5]            # TypeError: right side is a list
except TypeError as e:
    print(e)

{1, 2, 3, 4, 5}
unsupported operand type(s) for |: 'set' and 'list'


### Union 

All elements from both sets

In [89]:
a | b        # {1, 2, 3, 4, 5}
a.union(b)   # same

{1, 2, 3, 4, 5}

### Intersection

Only elements in both sets.

In [90]:
a & b              # {3}
a.intersection(b)  # same

{3}

### Difference

elements in a but not in b


In [91]:
a - b              # {1, 2}
a.difference(b)    # same

{1, 2}

### Symmetric Difference

Elements in either set, but not both

In [92]:
a ^ b                       # {1, 2, 4, 5}
a.symmetric_difference(b)   # same

{1, 2, 4, 5}

## Set Methods

Python sets have methods for adding, removing, and checking elements.

| Method             | Description                                   |
|--------------------|-----------------------------------------------|
| `.add(x)`          | Adds a single element                         |
| `.update(iterable)`| Adds multiple elements                        |
| `.copy()`       | Returns a shallow copy of the set  |
| `.remove(x)`       | Removes element, raises `KeyError` if missing |
| `.discard(x)`      | Removes element, silent if missing            |
| `.pop()`           | Removes and returns an arbitrary element      |
| `.clear()`         | Empties the set                               |
| `.issubset(s)`     | Returns `True` if all elements are in `s`     |
| `.issuperset(s)`   | Returns `True` if set contains all of `s`     |
| `.isdisjoint(s)`   | Returns `True` if no elements in common       |

### Adding elements:

- `add()`
- `update()`: adds multiple elements, similar to `dict.update()`. 

In [93]:
s = {1, 2, 3}
s.add(4)            # {1, 2, 3, 4}
s.update([5, 6])    # {1, 2, 3, 4, 5, 6}
s

{1, 2, 3, 4, 5, 6}

### Copy vs aliasing

Sets are mutable, so assignment shares the same object. Use `.copy()` when you need an independent set.

In [94]:
a = {1, 2}
b = a               # alias (same object)
c = a.copy()        # separate object

a.add(3)
print("a:", a)
print("b:", b)
print("c:", c)

a: {1, 2, 3}
b: {1, 2, 3}
c: {1, 2}


### Removing elements:

- `remove()`
- `discard()`
- `pop()`
- `clear()`

In [95]:
s.remove(3)     # raises KeyError if not found
print(s)        # {1, 2, 4, 5, 6}

s.discard(3)    # silent if not found — safer
print(s)        # {1, 2, 4, 5, 6}

s.pop()         # removes and returns an arbitrary element
print(s)        # which item is removed is arbitrary

s.clear()       # empties the set
print(s)        # set()

{1, 2, 4, 5, 6}
{1, 2, 4, 5, 6}
{2, 4, 5, 6}
set()


If you use `remove(()` in a `for` loop for a set, you may run into an error because Python creates an internal iterator that tracks position inside the set. The moment you call `s.remove()`, the set's structure changes and the iterator breaks. 

The solution to this problem is to iterate over a copy or just use a `while` loop.

In [ ]:
%%expect RuntimeError

s = {1, 2, 3,}
for item in s:
    s.remove(item)
    print(f"Processing {item}")

Processing 1


RuntimeError: Set changed size during iteration

In [ ]:
for item in s.copy():   # iterate over copy
    s.remove(item)      # modify original safely
    print(f"Processing {item}")
s

Processing 2
Processing 3


set()

#### Comparing `pop()`

`s.pop()` is useful when you want to process and consume a set one element at a time, and you don't care which element you get. 

In the example below, using `s.remove()` may raise `KeyError`.

In [96]:
# process all items, consuming the set
s = {1, 2, 3}
while s:
    item = s.pop()
    print(f"Processing {item}")
# s is now empty

Processing 1
Processing 2
Processing 3


| | `list.pop()` | `dict.pop()` | `set.pop()` |
|---|---|---|---|
| **Removes** | by index | by key | arbitrary element |
| **Returns** | removed value | removed value | removed value |
| **Default arg** | index (default `-1`) | key + optional default | none |
| **If not found** | `IndexError` | `KeyError` (or default) | `KeyError` |
| **Predictable?** | yes (by index) | yes (by key) | no (arbitrary item) |

### Checking membership:

In [99]:
s = {1, 2, 3}

print(3 in s)         # True
print(3 not in s)     # False

True
False


### Subset & superset:

In [100]:
a = {1, 2}
b = {1, 2, 3}

a.issubset(b)    # True — all of a is in b
b.issuperset(a)  # True — b contains all of a
a.isdisjoint(b)  # False — they share elements

False

Operator equivalents for subset/superset are often used in real code:

In [101]:
a = {1, 2}
b = {1, 2, 3}

print(a < b)    # proper subset
print(a <= b)   # subset
print(b > a)    # proper superset
print(b >= a)   # superset
print(a <= a)   # True (same set is subset of itself)
print(a < a)    # False (not a proper subset)

True
True
True
True
True
False


### In-place set operations

These methods modify the original set instead of creating a new one:

In [102]:
x = {1, 2, 3}
y = {3, 4, 5}

x.update(y)                        # union in-place
print("update:", x)

x = {1, 2, 3}
x.intersection_update(y)
print("intersection_update:", x)

x = {1, 2, 3}
x.difference_update(y)
print("difference_update:", x)

x = {1, 2, 3}
x.symmetric_difference_update(y)
print("symmetric_difference_update:", x)

update: {1, 2, 3, 4, 5}
intersection_update: {3}
difference_update: {1, 2}
symmetric_difference_update: {1, 2, 4, 5}


For in-place change methods for the common data structures, here is a comparison table. Tuple and strings are immutable, so there's no methods available for the purpose.

| Method                              | List | Dict | Set | Tuple | String |
|-------------------------------------|------|------|-----|-------|--------|
| `.append(x)`                        | ✓    |      |     |       |        |
| `.insert(i, x)`                     | ✓    |      |     |       |        |
| `.extend(iter)`                     | ✓    |      |     |       |        |
| `.update(other)`                    |      | ✓    | ✓   |       |        |
| `.remove(x)`                        | ✓    |      | ✓   |       |        |
| `.pop()`                            | ✓    | ✓    | ✓   |       |        |
| `.popitem()`                        |      | ✓    |     |       |        |
| `.add(x)`                           |      |      | ✓   |       |        |
| `.discard(x)`                       |      |      | ✓   |       |        |
| `.sort()`                           | ✓    |      |     |       |        |
| `.reverse()`                        | ✓    |      |     |       |        |
| `.setdefault(k, v)`                 |      | ✓    |     |       |        |
| `.clear()`                          | ✓    | ✓    | ✓   |       |        |
| `.intersection_update(t)`           |      |      | ✓   |       |        |
| `.difference_update(t)`             |      |      | ✓   |       |        |
| `.symmetric_difference_update(t)`   |      |      | ✓   |       |        |

## Frozensets

A frozenset is an immutable version of a set. Once created, elements cannot be added or removed.

In [111]:
%%expect AttributeError

fs = frozenset({1, 2, 3})
fs.add(4)     # AttributeError — frozensets are immutable

AttributeError: 'frozenset' object has no attribute 'add'

### Creating a frozenset

The syntax for creating a frozenset is:

```python
frozenset()            # empty frozenset
frozenset(iterable)    # from any iterable (list, set, string, etc.)
```

In [112]:
fs = frozenset([1, 2, 3])
print(fs)                   # frozenset({1, 2, 3})
fs = frozenset("hello")
print(fs)                   # element order may vary

frozenset({1, 2, 3})
frozenset({'l', 'e', 'h', 'o'})


### Why use a frozenset?

Because it is immutable, a frozenset is hashable, which means it can be: 

- used as a dictionary key, or 
- stored inside another set.

In [105]:
# As a dict key
d = {frozenset({1, 2}): "pair"}

# Inside a set
s = {frozenset({1, 2}), frozenset({3, 4})}

| Feature          | set               | frozenset              |
|------------------|-------------------|------------------------|
| Mutable          | Yes               | No                     |
| Hashable         | No                | Yes                    |
| Can be dict key  | No                | Yes                    |
| Set operations   | All               | Non-mutating only      |

## Performance & Best Practices

When you store a key in a dictionary or an element in a set, Python calls hash() on it to compute an integer, then uses that integer to determine where to store the value in memory. This is called a **hash table**.

The benefit is that lookups are usually O(1) on average: Python jumps directly to a bucket via hash instead of scanning the whole collection. In the worst case (many collisions), lookup can degrade toward O(n).

In [106]:
# O(n) — scans the entire list
"apple" in ["apple", "banana", "cherry"]

# Average O(1) — jumps directly via hash bucket
"apple" in {"apple", "banana", "cherry"}

True

**O(n)** and **O(1)** come from **Big O notation**, which describes how the **speed** of an operation scales with the **size** of the data. The full treatment of Big O comes in the Algorithms chapter, where sorting and searching will be covered.

For now, just know that:
- **Average O(1)**: hash-table lookups in `dict`/`set` are constant time on average, because Python uses the hash to jump to a bucket.
- **Worst-case O(n)**: if many keys collide into the same bucket, Python may need to scan multiple entries.
- **O(n)**: **linear time**. The operation gets slower as the collection grows. A list search is O(n) because Python checks each element one by one until it finds a match:

In [107]:
d = {"apple": 1, "banana": 2, "cherry": 3}
d["apple"]   # same speed whether dict has 3 or 3 million items

1

In [108]:
"apple" in ["apple", "banana", "cherry"]  # checks up to n items

True

In short, 

- **O(1)** is like looking up a word in a dictionary: you go directly to the page
- **O(n)** is like finding a word in a novel: you read until you find it

### Choosing the Right Structure

| Need                                   | Use         |
|----------------------------------------|-------------|
| Ordered collection, duplicates allowed | `list`      |
| Fixed data, multiple return values     | `tuple`     |
| Key-value lookup                       | `dict`      |
| Unique items, membership testing       | `set`       |
| Hashable set (dict key, nested set)    | `frozenset` |

## Set Comprehensions

A **set comprehension** builds a set from an expression, using the same syntax as a list comprehension but with curly braces `{}`. Because the result is a set, duplicates are automatically discarded.

The syntax for creating sets is:

```python
{expression for item in iterable}
{expression for item in iterable if condition}
```

In [115]:
# Create a set of squares
square_set = {x**2 for x in range(-3, 4)}
print("Square set:", square_set)

# Remove duplicates and transform
sentence = "the quick brown fox jumps over the lazy dog"
unique_lengths = {len(word) for word in sentence.split()}
print("Unique word lengths:", unique_lengths)

# Filter vowels
vowels = {char.lower() for char in "Hello World" if char.lower() in 'aeiou'}
print("Vowels found:", vowels)

Square set: {0, 9, 4, 1}
Unique word lengths: {3, 4, 5}
Vowels found: {'e', 'o'}


### Comparing with `dict` Comprehension

In [110]:
# Set comprehension — curly braces, no colon
{x**2 for x in range(5)}

# Dict comprehension — curly braces WITH colon
{x: x**2 for x in range(5)}

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}